# Assignment 03: Customer Churn Prediction with a Tuned Neural Network

**Student:** Arslan Hashmi  
**Internship ID:** ZYNVEX-CERT-0486  
**Institution:** National University of Technology (NUTECH)  
**Company:** Zynvex Solutions  
**Track:** AI/ML Internship | Week 3  
**GitHub:** https://github.com/arslannhashmi/AIML-Internship-Arslan_Hashmi

---

This is an end-to-end **deep learning project** for predicting **customer churn** using the real-world **Telco Customer Churn** dataset.  
The workflow covers data loading & cleaning, preprocessing for neural networks, baseline model training, model tuning (Dropout + Early Stopping + Batch Normalization), proper evaluation, model saving, and a short comparison of CNN vs RNN.

## 1. Load, Explore & Clean the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries loaded successfully')
print('TensorFlow version:', tf.__version__)

In [ ]:
# Upload the CSV in Colab using the file widget, or place it in the working directory
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')  # or 'Telco-Customer-Churn.csv'
print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
df.head()

In [ ]:
print('Missing values (null):')
print(df.isnull().sum())
print('\nBlank TotalCharges entries:', (df['TotalCharges'] == ' ').sum())
print('\nChurn distribution:')
print(df['Churn'].value_counts())
print('\nChurn proportion:')
print(df['Churn'].value_counts(normalize=True).round(3))

In [ ]:
# TotalCharges is stored as text and has a few blank entries → convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('NaNs after conversion:', df['TotalCharges'].isna().sum())

# Fill the few missing TotalCharges with the median
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Drop customerID (not useful for prediction)
df = df.drop('customerID', axis=1)
print('Shape after cleaning:', df.shape)

**Class Balance:** Churn is **imbalanced** (~73.5% No vs ~26.5% Yes).  
Accuracy alone can be misleading — a model that always predicts "No" would already achieve ~73.5% accuracy.

In [ ]:
# Visualize churn vs contract type and tenure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[0], palette='Set2')
axes[0].set_title('Churn by Contract Type')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[1], palette='Set2')
axes[1].set_title('Tenure by Churn Status')

plt.tight_layout()
plt.show()

**Observation:** Month-to-month contracts and low-tenure customers show much higher churn rates — these are strong predictive signals.

## 2. Preprocess for a Neural Network

In [ ]:
# Separate target
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop('Churn', axis=1)

# Identify categorical and numeric columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Categorical columns:', cat_cols)
print('Numeric columns:', num_cols)

In [ ]:
# Encoding strategy:
# - Binary categoricals → map to 0/1
# - Multi-class categoricals → One-Hot Encoding
# Justification: One-hot avoids false ordinal relationships for unordered categories.

binary_map = {
    'gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1}
}

for col, mapping in binary_map.items():
    X[col] = X[col].map(mapping)

# One-hot encode remaining multi-class columns
multi_cols = [c for c in cat_cols if c not in binary_map]
X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

print('Final feature shape:', X.shape)
X.head()

In [ ]:
# Scale numeric features (important for neural networks)
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Train-test split (stratified because of class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

## 3. Build & Train a Baseline Model

In [ ]:
input_dim = X_train.shape[1]

baseline_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='Baseline_NN')

baseline_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

baseline_model.summary()

In [ ]:
history_base = baseline_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    verbose=1
)

In [ ]:
# Training / Validation loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_base.history['loss'], label='Train Loss')
axes[0].plot(history_base.history['val_loss'], label='Val Loss')
axes[0].set_title('Baseline – Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_base.history['accuracy'], label='Train Acc')
axes[1].plot(history_base.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Baseline – Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

base_loss, base_acc = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f'Baseline Test Accuracy: {base_acc:.4f}')

## 4. Tune Your Model

Tuning techniques used:
1. **Dropout** – reduce overfitting
2. **Batch Normalization** – stabilize and speed up training
3. **Early Stopping** – stop when validation loss stops improving
4. L2 regularization + slightly deeper architecture

In [ ]:
tuned_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
], name='Tuned_NN')

tuned_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tuned_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

history_tuned = tuned_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=60,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_tuned.history['loss'], label='Train Loss')
axes[0].plot(history_tuned.history['val_loss'], label='Val Loss')
axes[0].set_title('Tuned Model – Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_tuned.history['accuracy'], label='Train Acc')
axes[1].plot(history_tuned.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Tuned Model – Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

tuned_loss, tuned_acc = tuned_model.evaluate(X_test, y_test, verbose=0)
print(f'Tuned Test Accuracy: {tuned_acc:.4f}')
print(f'\nBefore/After Comparison:')
print(f'  Baseline Test Accuracy : {base_acc:.4f}')
print(f'  Tuned Test Accuracy    : {tuned_acc:.4f}')
print(f'  Improvement            : {(tuned_acc - base_acc)*100:.2f} percentage points')

## 5. Evaluate Properly

In [ ]:
y_pred_prob = tuned_model.predict(X_test).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

print('Classification Report (Tuned Model):')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix – Tuned Model')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

**Why accuracy alone can be misleading:**  
Because the dataset is imbalanced (~73.5% non-churners), a naïve model that always predicts “No Churn” would already score ~73.5% accuracy. Precision, recall and F1-score for the minority (Churn) class give a much clearer picture of real predictive power.

## 6. Save & Predict

In [ ]:
# Save the best (tuned) model
tuned_model.save('churn_model.keras')
print('Model saved as churn_model.keras')

In [ ]:
def predict_churn(model_path, customer_features):
    """
    Load a saved model and predict churn probability for one customer.
    customer_features must be a 1-D array / list with the same number of
    features used during training (already encoded & scaled).
    """
    model = keras.models.load_model(model_path)
    features = np.array(customer_features).reshape(1, -1)
    prob = model.predict(features, verbose=0)[0][0]
    label = 'Churn' if prob >= 0.5 else 'No Churn'
    return {'probability': float(prob), 'prediction': label}


# Example: take one real test sample as a hypothetical customer
sample = X_test.iloc[0].values
result = predict_churn('churn_model.keras', sample)
print('Hypothetical customer prediction:')
print(f"  Churn probability : {result['probability']:.4f}")
print(f"  Predicted class   : {result['prediction']}")

## 7. Short Answer: CNN vs. RNN

**When to use a CNN instead of a Dense network:**  
CNNs excel when the input has a clear spatial or local structure (images, spectrograms, or any grid-like data). Convolutional filters automatically learn local patterns (edges, textures) and are translation-equivariant, making them far more parameter-efficient than a fully-connected network for high-dimensional spatial inputs.  
**Example:** Image classification (e.g., detecting tumors in medical scans) or any computer-vision task.

**When to use an RNN instead of a Dense network:**  
RNNs (and their modern variants LSTM/GRU) are designed for sequential data where order and temporal dependencies matter. They maintain a hidden state that carries information across time steps.  
**Example:** Sentiment analysis of a product review, machine translation, or time-series forecasting of stock prices / sensor readings.

In short: use **CNNs** for spatial structure, **RNNs** for sequential/temporal structure, and plain **Dense networks** when features are independent tabular attributes (as in this churn dataset).

---
## Summary

| Task | What was done |
|------|---------------|
| 1. Load & Clean | Handled TotalCharges blanks, dropped customerID, confirmed class imbalance |
| 2. Preprocess | Binary + one-hot encoding, StandardScaler, stratified split |
| 3. Baseline | 2-hidden-layer Dense network + loss/accuracy curves |
| 4. Tuning | Dropout + BatchNorm + EarlyStopping + L2 → improved test accuracy |
| 5. Evaluation | Precision / Recall / F1 + Confusion Matrix; accuracy alone is misleading |
| 6. Deployment | `churn_model.keras` saved + prediction helper function |
| 7. CNN vs RNN | Clear use-case distinction |

**Files produced:**
- `Assignment3_ArslanHashmi.ipynb`
- `churn_model.keras`

**Completed by:** Arslan Hashmi  
**Date:** August 2026